                                Sentiment Analysis with an Recurrent Neural Networks (RNN)                         
                                                            

## Sentiment Analysis with an Recurrent Neural Networks (RNN)


Recurrent Neural Networks (RNNs) are used in sequence tasks such as sentiment analysis due to their ability to capture context from sequential data. In this article we will be apply RNNs to analyze the sentiment of customer reviews from Swiggy food delivery platform. The goal is to classify reviews as positive or negative for providing insights into customer experiences.

We will conduct a Sentiment Analysis using the TensorFlow framework:

### 1. Importing Libraries and Dataset


In [1]:
import pandas as pd
import numpy as np
import re  
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

### 2. Loading Dataset

We will be using swiggy dataset of customer reviews.

In [2]:
dataset = "https://raw.githubusercontent.com/itsluckysharma01/Datasets/refs/heads/main/swiggy.csv"

data = pd.read_csv(dataset)
print("Data loaded successfully!")

Data loaded successfully!


In [3]:
data.head(4)

,ID,Area,City,Restaurant Price,Avg Rating,Total Rating,Food Item,Food Type,Delivery Time,Review
0,1,Suburb,Ahmedabad,600,4.2,6198,Sushi,Fast Food,30-40 min,"Good, but nothing extraordinary."
1,2,Business District,Pune,200,4.7,4865,Pepperoni Pizza,Non-Vegetarian,50-60 min,"Good, but nothing extraordinary."
2,3,Suburb,Bangalore,600,4.7,2095,Waffles,Fast Food,50-60 min,Late delivery ruined it.
3,4,Business District,Mumbai,900,4.0,6639,Sushi,Vegetarian,50-60 min,Best meal I've had in a while!


In [4]:
data.columns

Index(['ID', 'Area', 'City', 'Restaurant Price', 'Avg Rating', 'Total Rating',
       'Food Item', 'Food Type', 'Delivery Time', 'Review'],
      dtype='object')

### 3. Text Cleaning and Sentiment Labeling

We will clean the review text, create a sentiment label based on ratings and remove any missing values.

In [7]:
data["Review"] = data["Review"].str.lower()
data["Review"] = data["Review"].replace(r'[^a-z0-9\s]', '', regex=True)

data["Sentiment"] = data["Avg Rating"].apply(lambda x: 1 if x >= 3.5 else 0)
data.dropna(inplace=True)

### 4. Tokenization and Padding

We will prepare the text data by tokenizing and padding it and extract the target sentiment labels. Tokenizer converts words into integer sequences and padding ensures all input sequences have the same length (max_length).

In [8]:
data.head(3)

,ID,Area,City,Restaurant Price,Avg Rating,Total Rating,Food Item,Food Type,Delivery Time,Review,Sentiment
0,1,Suburb,Ahmedabad,600,4.2,6198,Sushi,Fast Food,30-40 min,good but nothing extraordinary,1
1,2,Business District,Pune,200,4.7,4865,Pepperoni Pizza,Non-Vegetarian,50-60 min,good but nothing extraordinary,1
2,3,Suburb,Bangalore,600,4.7,2095,Waffles,Fast Food,50-60 min,late delivery ruined it,1


In [9]:
data.tail(3)

,ID,Area,City,Restaurant Price,Avg Rating,Total Rating,Food Item,Food Type,Delivery Time,Review,Sentiment
7997,7998,Tech Park,Chennai,900,4.5,4645,Mango Shake,Fast Food,30-40 min,nothing special but edible,1
7998,7999,Old Town,Delhi,500,4.2,3218,Grilled Cheese,Non-Vegetarian,50-60 min,it was okay,1
7999,8000,Downtown,Delhi,400,4.5,7739,Burrito,Vegan,30-40 min,delicious and fresh,1


In [11]:
max_features = 5000
max_length = 200

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(data["Review"])

X = pad_sequences(tokenizer.texts_to_sequences(data["Review"]), maxlen=max_length)
y = data["Sentiment"].values

### 5. Splitting the Data

We will split the data into training, validation and test sets while maintaining the class distribution.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_test, y_test, test_size=0.1, random_state=42, stratify=y_test)


### 6. Building RNN Model

We will build and compile a simple RNN model for binary sentiment classification.

In [13]:
model = Sequential([
    Embedding(input_dim=max_features, output_dim=16, input_length=max_length),
    SimpleRNN(32, activation='tanh', return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

c:\Users\PANDIT JI\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### 7. Training and Evaluating Model

We will train the model on training data, validate it during training, then evaluate its performance on test data.

In [14]:
history = model.fit(X_train, y_train, epochs=15, batch_size=32, validation_data=(X_val, y_val), verbose=2)
score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {score[0]:.4f}, Test Accuracy: {score[1]:.4f}")


Epoch 1/15
45/45 - 3s - 66ms/step - accuracy: 0.8375 - loss: 0.4488 - val_accuracy: 0.8562 - val_loss: 0.4782
Epoch 2/15
45/45 - 2s - 50ms/step - accuracy: 0.8569 - loss: 0.4168 - val_accuracy: 0.8562 - val_loss: 0.4249
Epoch 3/15
45/45 - 1s - 28ms/step - accuracy: 0.8569 - loss: 0.4078 - val_accuracy: 0.8562 - val_loss: 0.4216
Epoch 4/15
45/45 - 2s - 34ms/step - accuracy: 0.8569 - loss: 0.4058 - val_accuracy: 0.8562 - val_loss: 0.4229
Epoch 5/15
45/45 - 1s - 27ms/step - accuracy: 0.8569 - loss: 0.4042 - val_accuracy: 0.8562 - val_loss: 0.4246
Epoch 6/15
45/45 - 1s - 30ms/step - accuracy: 0.8569 - loss: 0.4036 - val_accuracy: 0.8562 - val_loss: 0.4264
Epoch 7/15
45/45 - 1s - 28ms/step - accuracy: 0.8569 - loss: 0.4040 - val_accuracy: 0.8562 - val_loss: 0.4289
Epoch 8/15
45/45 - 1s - 26ms/step - accuracy: 0.8569 - loss: 0.4030 - val_accuracy: 0.8562 - val_loss: 0.4301
Epoch 9/15
45/45 - 1s - 31ms/step - accuracy: 0.8569 - loss: 0.4023 - val_accuracy: 0.8562 - val_loss: 0.4302
Epoch 10/1

### 8. Predicting Sentiment

We will create a function to preprocess a single review, predict its sentiment and display the result.


In [17]:
def predict_sentiment(review_text):
    text = review_text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length)

    prediction = model.predict(padded)[0][0]
    return f"{'Positive' if prediction >= 0.5 else 'Negative'} (Probability: {prediction:.2f})"


sample_review = "The food was great."
print(f"Review: {sample_review}")
print(f"Sentiment: {predict_sentiment(sample_review)}")

Review: The food was great.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
Sentiment: Positive (Probability: 0.83)


In [21]:
def predict_sentiment(review_text):
    text = review_text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length)

    prediction = model.predict(padded)[0][0]
    return f"{'Positive' if prediction >= 0.5 else 'Negative'} (Probability: {prediction:.2f})"


sample_review = "wrost food ever"
print(f"Review: {sample_review}")
print(f"Sentiment: {predict_sentiment(sample_review)}")

Review: wrost food ever
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
Sentiment: Positive (Probability: 0.83)
